### Initialize instruments

In [ ]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, LeCroyScope
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

In [ ]:
# set up readout and plotting
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [ ]:
# open list of available instruments
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
# connect to BNC 575 pulse generator (for controlling experimental time sequence)
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
# connect to SR400 gated photon counter (for performing Rydberg atom counting)
sr400 = SR400.SR400("GPIB0::23::INSTR", timeout = 5000)

### Set compensation voltages and define timing sequence, set up photon counter

In [ ]:
# define timing sequence

############################## current pulsed sequence ##############################
# pulse_arrangement = [["A", 1300e-6, 0, 2, 1],                 # MOT AOM
#                      [["B", 15e-6, 40e-6, 2, 1],              # 780 AOM
#                      ["C", 15e-6, 40e-6, 6, -1]],             # 480 switch
#                      ["D", 26e-6, 0, 6, 1],                   # DEI switch/gated photon counter
#                     ]

pulse_arrangement = [
                    [["D", 26e-06, 1355e-06, 6, 1],             # AWG; ampl changed from 6 V
                    ["E", 26e-06, 1355e-06, 6, 1]]              # gated photon counter
                    ]
T = 1600e-6  # sequence period; changed from 1500 us

cycle_number = 30

# photon counter gate width and delay
gate_width = 130e-6 # changed from 20 us
gate_delay = 3e-6  # unchanged

notes = {"gate_width": gate_width, "gate_delay": gate_delay}

450 V @ T = 200 us => 1500 V @ T = (1500/450)*200 ~ 700 us (works)

Experimentally observed smallest T: 530 us (saw 1.0 mA draw from power supply; drops to 0.08 mA when BNC is turned off)

No apparent dependence on other channels; increased blue exc duration from 15 to 150 to 300 us and saw no change in current draw (still 1.0 mA @ 530 us total sequence period)

Preliminary conclusion: The DEI pulse width has to be less than ~5% (maybe closer to 4.9%) of the total sequence period to avoid reaching current limit on HV supply.

In [ ]:
# set up the photon counter; count_preset changed from 100 us
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 140e-6, count_period_num = cycle_number, 
                  dwell = 0, sourceA = "INPUT1", sourceB = "INPUT2", gate_A_mode = "FIXED",
                  gate_A_delay = gate_delay, gate_A_width = gate_width, gate_B_mode = "CW")

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

### Find the Rydberg resonance

### Collect oscilloscope traces

In [ ]:
# connect to LeCroy 9354AM oscilloscope (for displaying DEI pulse and Rydberg counts for both states)
lecroy = LeCroyScope.LeCroyScope('GPIB0::3::INSTR', timeout = 20000)
lecroy.setup_transfer()

In [ ]:
# collection parameters
# N = 500 # number of scope sweeps averaged
V_mcp = -550 # MCP front voltage [V]
# V_tip = 500 # SSFI voltage on tip electrode [V]
dist = 3.0 # approximate distance between TIP and MOT [mm]
DEI_width = pulse_arrangement[2][1] # DEI pulse width [us]

In [ ]:
t_AWG, v_AWG, info_AWG = lecroy.read_waveform("C2")
t_AMP, v_AMP, info_AMP = lecroy.read_waveform("C3")

In [ ]:
# save results for DEI pulse
notes = {"DEI pulse width (us)": DEI_width, "Maximum AWG voltage (V)": 0.25}
ps.Plot2D(x = t_AMP, y = None, Z = v_AMP,
          xlabel = "Time (s)", zlabel = "DEI pulse (V)", notes = notes)

In [ ]:
# run to disconnect from LeCroy (optional)
lecroy.close()

### Shut down instrumentation

In [ ]:
# reset voltages to zero
vset = np.array([0.0,0.0,0.0])
mcbox.set_voltage_Nchan(channels = [0,1,6,7,10,11], voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# save current stepper motor position
weeder.save(header = "A", query = True)

In [ ]:
# OPTIONAL: disables pulses from BNC box (can also be done by pressing RUN/STOP button or individual channel buttons on device)
bnc575.disable_all()

In [ ]:
# OPTIONAL: disarms the pulse generator (can also be done by pressing RUN/STOP button on device)
bnc575.disarm_all()

In [ ]:
# OPTIONAL: used to reset photon counter (can also be done by pressing STOP button on device)
sr400.count_reset()

In [ ]:
# disconnect from pulse generator
bnc575.pyvisa.close()

In [ ]:
# disconnect from SRS photon counter
sr400.pyvisa.close()

In [ ]:
# disconnect from oscillscope
lecroy.close()